# Gene Analysis Notebook

This notebook loads the generated Mytho-Toon Genome using PySpark and displays the results.

Use the super pyenv to run this notebook; check README.md for details.


In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, size
import pandas as pd

# Ensure we can import our modules
# sys.path.append(os.path.abspath("super-services/src"))

from super.core.runtime import *

from super.core import utils
from super.apps.generate_powers.models import MutatedGene

from super.core.display import display_scrollable_dataframe

In [2]:
# Initialize Spark Session
bootstrap_spark_env()

# set memory above default
spark = (SparkSession.builder
.appName("VCP Low-Level Catalog")
.config("spark.executor.memory", "16g")
.config("spark.driver.memory", "16g")
.getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 01:56:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/19 01:56:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:

# Load App Configuration to find paths
conf = utils.get_app_conf("generate_powers")
stage_root = conf.get_string("stage_root")

print(f"Project Stage Root: {stage_root}")

Project Stage Root: /home/gideon/tmp/super_powers/data


In [4]:
# Defined path to genes
genes_path = os.path.join(stage_root, "generated_genome", "*", "*.json")
print(f"Reading genes from: {genes_path}")

# Load JSONs into DataFrame
raw_sdf = spark.read.option("multiline", "true").json(genes_path)

# Show Schema
raw_sdf.printSchema()

Reading genes from: /home/gideon/tmp/super_powers/data/generated_genome/*/*.json


26/01/19 01:56:37 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/gideon/tmp/super_powers/data/generated_genome/*/*.json.
java.io.FileNotFoundException: File /home/gideon/tmp/super_powers/data/generated_genome/*/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Res

root
 |-- confidence: double (nullable = true)
 |-- failure_mode: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_role: string (nullable = true)
 |-- mutation_class: string (nullable = true)
 |-- primary_seeds: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- seed: string (nullable = true)
 |    |    |-- weight: double (nullable = true)
 |-- regulated_genes: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- effect: string (nullable = true)
 |    |    |-- gene_id: string (nullable = true)
 |    |    |-- strength: double (nullable = true)
 |-- secondary_seeds: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- seed: string (nullable = true)
 |    |    |-- weight: double (nullable = true)
 |-- side_effect_profile: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- probability: double (nullable = true)
 |    |    |-- se

In [5]:
# Analysis: Show top level fields
display_sdf = raw_sdf.select(
    col("gene_id"),
    col("gene_role"),
    col("mutation_class"),
    col("failure_mode"),
    col("confidence"),
    size(col("regulated_genes")).alias("num_links"),
    size(col("side_effect_profile")).alias("num_side_effects")
)

display_sdf.show(20, truncate=False)

+----------+---------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------+----------------+
|gene_id   |gene_role|mutation_class        |failure_mode                                                                                                                                                      |confidence|num_links|num_side_effects|
+----------+---------+----------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------+----------------+
|HAMMER-321|hub      |toon_causality_break  |Accessing Hammerspace while influenced by Cosmic Empowerment causes a collapse in local probability fields, leading to unpredictable object dynamics.             |0.83      |30       |4               |
|OPTICE-847|

In [6]:
raw_df = raw_sdf.toPandas()

display_scrollable_dataframe(raw_df)

In [ ]:
# Count by Role
raw_sdf.groupBy("gene_role").count().show()